# FM-PCC Colab Workflow

---

## Run Order
1. Mount Drive
2. Clone/Update FM-PCC
3. Boost startup cache wiring
4. Install Miniconda + Create `FMPCC` env
5. Install D3IL
6. Install requirements with pinned compatibility
7. Set runtime env variables
8. Optional W&B relogin
9. Verify dependencies
10. Prepare dataset
11. Smoke test
12. Full train
13. Eval + visualization
14. Archive logs



## 1) Mount Google Drive



In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

FMPCC_ROOT = '/content/drive/MyDrive/FMPCC'
REPO = f'{FMPCC_ROOT}/FM-PCC'
os.makedirs(FMPCC_ROOT, exist_ok=True)

print('Drive mounted')
print('Repo path:', REPO)



## 2) Clone or Update FM-PCC



In [ ]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
REPO="$ROOT/FM-PCC"
UPDATE_REPO="${UPDATE_REPO:-1}"
OVERWRITE_LOCAL_CHANGES="${OVERWRITE_LOCAL_CHANGES:-0}"

mkdir -p "$ROOT"
cd "$ROOT"

if [ ! -d "$REPO/.git" ]; then
  git clone --recurse-submodules https://github.com/ghubliming/FM-PCC.git
else
  cd "$REPO"
  if [ "$UPDATE_REPO" != "1" ]; then
    echo "Repo update disabled (UPDATE_REPO=$UPDATE_REPO). Using existing local checkout."
    echo "Repo ready: $REPO"
    exit 0
  fi

  CHANGED_FILES="$(git status --porcelain)"
  if [ -n "$CHANGED_FILES" ]; then
    echo "WARNING: Local changes detected in repo."
    echo "Changed files:"
    echo "$CHANGED_FILES"

    if [ "$OVERWRITE_LOCAL_CHANGES" = "1" ]; then
      echo "OVERWRITE_LOCAL_CHANGES=1 -> discarding local changes and untracked files"
      git reset --hard HEAD
      git clean -fd
    else
      echo "Skipping git pull to protect local changes."
      echo "Set OVERWRITE_LOCAL_CHANGES=1 to force overwrite on next run."
      echo "Repo ready: $REPO"
      exit 0
    fi
  fi

  git fetch origin
  git reset --hard origin/main
  git submodule update --init --recursive
fi

echo "Repo ready: $REPO"



## 3) Boost Startup Cache Wiring

Keeps the original `/content/miniconda3` path logic, but maps it to Drive so restarts can reuse the same conda env and pip cache.



In [ ]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
PIP_CACHE="$ROOT/.pip-cache"

mkdir -p "$ROOT" "$PIP_CACHE"

if [ -L "$RUNTIME_CONDA" ]; then
  rm -f "$RUNTIME_CONDA"
elif [ -d "$RUNTIME_CONDA" ]; then
  rm -rf "$RUNTIME_CONDA"
fi

ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"

echo "Runtime conda path mapped to: $PERSIST_CONDA"
echo "Persistent pip cache path: $PIP_CACHE"



## 4) Install Miniconda and Create Env

Keeps Python pinned to 3.10 for compatibility with project dependencies.



In [ ]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
CONDA_BIN="$PERSIST_CONDA/bin/conda"

# Ensure the runtime path points to the persistent conda directory.
if [ -L "$RUNTIME_CONDA" ]; then
  LINK_TARGET="$(readlink -f "$RUNTIME_CONDA" || true)"
  if [ "$LINK_TARGET" != "$PERSIST_CONDA" ]; then
    rm -f "$RUNTIME_CONDA"
    ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
  fi
elif [ ! -e "$RUNTIME_CONDA" ]; then
  ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
fi

if [ ! -x "$CONDA_BIN" ] || ! "$CONDA_BIN" --version >/dev/null 2>&1; then
  # Clean partial/corrupt install so installer can create a fresh tree.
  rm -rf "$PERSIST_CONDA"
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
  bash /content/miniconda.sh -b -p "$PERSIST_CONDA" -u
fi

"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! "$CONDA_BIN" env list | grep -q "^FMPCC "; then
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

if [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/python" ] || [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/pip" ]; then
  "$CONDA_BIN" remove -n FMPCC --all -y || true
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

"$PERSIST_CONDA/envs/FMPCC/bin/python" -V
"$PERSIST_CONDA/envs/FMPCC/bin/pip" --version



## 5) Install D3IL (Install Once + Verify)

Uses editable installs for both D3IL core and `gym_avoiding_env`, but skips reinstall when editable links already exist.



In [ ]:
%%bash
set -e

PIP="/content/miniconda3/envs/FMPCC/bin/pip"
REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
D3IL="$REPO/d3il"

if [ ! -d "$D3IL/.git" ]; then
  echo "d3il missing/incomplete -> recloning"
  rm -rf "$D3IL"
  git clone https://github.com/ALRhub/d3il.git "$D3IL"
fi

if "$PIP" freeze | grep -Fq "d3il/environments/d3il" && "$PIP" freeze | grep -Fq "d3il/envs/gym_avoiding_env"; then
  echo "D3IL editable installs already present; skipping reinstall"
else
  "$PIP" install -e "$D3IL/environments/d3il"
  "$PIP" install -e "$D3IL/environments/d3il/envs/gym_avoiding_env"
fi

echo "D3IL installed"



## 6) Install Requirements (Install Once + Verify)

Runs validation first and only installs when the environment is missing or inconsistent.



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
PIP="/content/miniconda3/envs/FMPCC/bin/pip"
PY="/content/miniconda3/envs/FMPCC/bin/python"
PIP_CACHE="/content/drive/MyDrive/FMPCC/.pip-cache"

mkdir -p "$PIP_CACHE"
cd "$REPO"

if "$PY" - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        importlib.import_module(p)
    except Exception as e:
        ok = False
        print(f'Missing/broken package: {p} ({type(e).__name__}: {e})')

import numpy
if int(numpy.__version__.split('.')[0]) >= 2:
    ok = False
    print(f'Invalid numpy version: {numpy.__version__}; expected 1.x for this workflow')

sys.exit(0 if ok else 2)
PY
then
  echo "Package validation passed; skipping requirements reinstall"
else
  echo "Package validation failed; installing requirements"
  PIP_CACHE_DIR="$PIP_CACHE" "$PIP" install -r requirements.txt
  "$PIP" check
fi

# Quick sanity check
"$PY" - <<'PY'
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("python:", __import__("sys").executable)
PY



## 7) Runtime Environment Variables

Includes W&B malformed service cleanup and Colab rendering settings.



In [ ]:
import os

FMPCC = '/content/drive/MyDrive/FMPCC/FM-PCC'
D3IL_ROOT = f'{FMPCC}/d3il'
GYM_AV = f'{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env'

existing_pp = os.environ.get('PYTHONPATH', '')
parts = [FMPCC, D3IL_ROOT, GYM_AV]
if existing_pp:
    parts.append(existing_pp)

os.environ['FMPCC'] = FMPCC
os.environ['PYTHONPATH'] = ':'.join(parts)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'agg'

for key in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(key, None)

os.chdir(FMPCC)
print('cwd:', os.getcwd())
print('PYTHONPATH:', os.environ['PYTHONPATH'])



## 8) Optional W&B Login



In [ ]:
import os
from pathlib import Path
import wandb

KEY_FILE = Path('/content/drive/MyDrive/FMPCC/.wandb_api_key')

if not KEY_FILE.exists():
    raise FileNotFoundError(
        f'Missing W&B key file: {KEY_FILE}. Create it with your API key on one line.'
    )

api_key = KEY_FILE.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'W&B key file is empty: {KEY_FILE}')

for k in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(k, None)

wandb.finish()
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_API_KEY'] = api_key

wandb.login(key=api_key, relogin=True)

print('W&B mode:', os.environ.get('WANDB_MODE'))
print('W&B key file:', KEY_FILE)
print('W&B key loaded:', f'***{api_key[-4:]}' if len(api_key) >= 4 else '***')



## 9) Full Verification

Validates import chain with the exact env interpreter used for training.



In [ ]:
%%bash
set -e

/content/miniconda3/envs/FMPCC/bin/python - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        m = importlib.import_module(p)
        v = getattr(m, '__version__', 'unknown')
        print(f'{p:20s} {v}')
    except Exception as e:
        ok = False
        print(f'{p:20s} NOT IMPORTABLE ({type(e).__name__}: {e})')

import numpy, torch
print('numpy pinned:', numpy.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

major = int(numpy.__version__.split('.')[0])
if major >= 2:
    ok = False
    print('ERROR: numpy 2.x detected, expected 1.26.4 for this workflow')

if not ok:
    sys.exit(2)
PY



## 10) Dataset Preparation (Avoiding)

### Option A: Use existing zip from old DPCC path
Keeps your original logic. This exits quickly if avoiding data already exists.



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
AVOIDING_DATA="$REPO/d3il/environments/dataset/data/avoiding/data"
DATA_ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A "$AVOIDING_DATA")" ]; then
  echo "avoiding data already present: $(ls "$AVOIDING_DATA" | wc -l) files"
  exit 0
fi

if [ ! -f "$DATA_ZIP" ]; then
  echo "dataset zip not found: $DATA_ZIP"
  echo "Skip this cell and use Option B below if needed."
  exit 1
fi

TMP="/content/avoiding_tmp"
rm -rf "$TMP"
mkdir -p "$TMP"
unzip -q "$DATA_ZIP" "avoiding/*" -d "$TMP"
mkdir -p "$REPO/d3il/environments/dataset/data/avoiding"
cp -r "$TMP/avoiding/." "$REPO/d3il/environments/dataset/data/avoiding/"
rm -rf "$TMP"

echo "avoiding dataset ready: $(ls "$AVOIDING_DATA" | wc -l) files"



### Option B: Download full D3IL dataset zip with gdown (only if Option A unavailable)



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
DATA_DIR="$REPO/d3il/environments/dataset/data"
ZIP_FILE="$DATA_DIR/dataset.zip"

if [ -f "$ZIP_FILE" ]; then
  echo "zip already exists: $ZIP_FILE"
  exit 0
fi

/content/miniconda3/envs/FMPCC/bin/pip install gdown -q
/content/miniconda3/envs/FMPCC/bin/python -m gdown \
  "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
  -O "$ZIP_FILE"

echo "downloaded zip: $ZIP_FILE"



## 11) Smoke Test Train

Short check before full run.



## 12) Full Train

Real-time streaming via `!python`.



In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC



## 13) Resume Training (Optional)



In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --use-wandb --auto-resume --wandb-project FMPCC



## 14) Evaluation and Results



Remember to edit the yaml in /config to choose seeds and must write_to_file: True


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/eval.py
!/content/miniconda3/envs/FMPCC/bin/python scripts/load_results.py



## 15) Visualization



In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/visualize_data_constraints.py



## 16) Archive Logs to Drive



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
STAMP="$(date +%Y%m%d_%H%M%S)"
OUT="/content/drive/MyDrive/FMPCC/runs_snapshot/$STAMP"

mkdir -p "$OUT"
if [ -d "$REPO/logs" ]; then
  cp -r "$REPO/logs" "$OUT/"
fi
if [ -d "$REPO/wandb" ]; then
  cp -r "$REPO/wandb" "$OUT/"
fi

echo "snapshot saved: $OUT"
